# Notebook 05 — Coverage Phase Diagram

This notebook combines the NMF recovery outputs from Notebook 03 and the SAE dilution outputs from Notebook 04 into shared phase-diagram metrics.

**Goal:** compare structure recovery across methods using common residue-manifold metrics.

Expected inputs in `data/`:
- an NMF summary CSV from Notebook 03, usually `nmf_recovery_summary.csv`
- an SAE summary CSV from Notebook 04, usually `sae_dilution_summary.csv`

Outputs:
- `data/coverage_phase_diagram.csv`
- `data/method_comparison_summary.csv`
- `figures/coverage_phase_diagram.svg`
- `figures/alignment_phase_diagram.svg`
- `figures/cost_vs_structure_quality.svg`
- `figures/method_regime_map.svg`

In [ ]:
# Setup

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]
N_LANES = len(VALID_LANES_MOD30)

def save_svg(fig, name):
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

In [ ]:
# Inspect available data files

data_files = sorted([f for f in os.listdir("data") if f.endswith(".csv")])
print("CSV files in data/:")
for f in data_files:
    print(" -", f)

In [ ]:
# Robust CSV discovery

def find_csv(preferred_name, required_keywords, fallback_keywords=None):
    """Find a CSV in data/ by preferred filename first, then by keywords."""
    files = sorted([f for f in os.listdir("data") if f.endswith(".csv")])

    if preferred_name in files:
        return preferred_name

    matches = []
    for f in files:
        lower = f.lower()
        if all(k.lower() in lower for k in required_keywords):
            matches.append(f)

    if not matches and fallback_keywords:
        for f in files:
            lower = f.lower()
            if all(k.lower() in lower for k in fallback_keywords):
                matches.append(f)

    if not matches:
        raise FileNotFoundError(
            "Could not find a CSV matching "
            f"preferred={preferred_name!r}, required_keywords={required_keywords}. "
            f"Available CSVs: {files}"
        )

    # Prefer files with "recovery" or "dilution" over component/feature summaries.
    priority_words = ["recovery", "dilution", "summary"]
    matches = sorted(
        matches,
        key=lambda f: sum(word in f.lower() for word in priority_words),
        reverse=True,
    )
    return matches[0]

nmf_file = find_csv(
    preferred_name="nmf_recovery_summary.csv",
    required_keywords=["nmf", "recovery"],
    fallback_keywords=["nmf", "summary"],
)

sae_file = find_csv(
    preferred_name="sae_dilution_summary.csv",
    required_keywords=["sae", "dilution"],
    fallback_keywords=["sae", "summary"],
)

print()
print("Using files:")
print("NMF:", nmf_file)
print("SAE:", sae_file)

nmf_raw = pd.read_csv(os.path.join("data", nmf_file))
sae_raw = pd.read_csv(os.path.join("data", sae_file))

print()
print("NMF columns:", list(nmf_raw.columns))
print("SAE columns:", list(sae_raw.columns))

In [ ]:
# Normalize NMF table

nmf = nmf_raw.copy()

# Notebook 03 used k as number of NMF components.
if "capacity" not in nmf.columns:
    if "k" in nmf.columns:
        nmf = nmf.rename(columns={"k": "capacity"})
    elif "n_components" in nmf.columns:
        nmf = nmf.rename(columns={"n_components": "capacity"})

# Normalize reconstruction column.
if "reconstruction_mse" not in nmf.columns:
    for candidate in ["mse", "mean_squared_error", "reconstruction_error"]:
        if candidate in nmf.columns:
            nmf = nmf.rename(columns={candidate: "reconstruction_mse"})
            break

# Normalize lane-mass column.
if "lane_mass_ratio" not in nmf.columns:
    if "mean_lane_mass_ratio" in nmf.columns:
        nmf = nmf.rename(columns={"mean_lane_mass_ratio": "lane_mass_ratio"})
    elif "alignment" in nmf.columns:
        nmf = nmf.rename(columns={"alignment": "lane_mass_ratio"})

required_nmf = ["capacity", "reconstruction_mse", "lane_mass_ratio"]
missing_nmf = [c for c in required_nmf if c not in nmf.columns]
if missing_nmf:
    raise ValueError(f"NMF CSV is missing required columns {missing_nmf}. Columns found: {list(nmf.columns)}")

nmf["method"] = "NMF"
nmf["topk"] = np.nan

# NMF sweep is expected to recover valid lanes compactly at/near k=8.
# If coverage is absent, define conservative coverage from capacity capped at 8.
if "coverage" not in nmf.columns:
    nmf["coverage"] = np.minimum(nmf["capacity"] / N_LANES, 1.0)

if "dead_features" not in nmf.columns:
    nmf["dead_features"] = 0

if "redundant_valid_features" not in nmf.columns:
    nmf["redundant_valid_features"] = np.maximum(nmf["capacity"] - N_LANES, 0)

nmf_keep = nmf[
    [
        "method",
        "capacity",
        "topk",
        "coverage",
        "lane_mass_ratio",
        "reconstruction_mse",
        "dead_features",
        "redundant_valid_features",
    ]
].copy()

nmf_keep.head()

In [ ]:
# Normalize SAE table

sae = sae_raw.copy()

if "capacity" not in sae.columns:
    if "hidden_dim" in sae.columns:
        sae = sae.rename(columns={"hidden_dim": "capacity"})
    elif "dictionary_size" in sae.columns:
        sae = sae.rename(columns={"dictionary_size": "capacity"})

if "reconstruction_mse" not in sae.columns:
    if "final_loss" in sae.columns:
        sae = sae.rename(columns={"final_loss": "reconstruction_mse"})
    elif "loss" in sae.columns:
        sae = sae.rename(columns={"loss": "reconstruction_mse"})

if "lane_mass_ratio" not in sae.columns:
    if "mean_lane_mass_ratio" in sae.columns:
        sae = sae.rename(columns={"mean_lane_mass_ratio": "lane_mass_ratio"})

if "redundant_valid_features" not in sae.columns:
    if "redundancy" in sae.columns:
        sae = sae.rename(columns={"redundancy": "redundant_valid_features"})
    else:
        sae["redundant_valid_features"] = 0

if "dead_features" not in sae.columns:
    sae["dead_features"] = 0

if "coverage" not in sae.columns:
    if "unique_valid_lanes" in sae.columns:
        sae["coverage"] = sae["unique_valid_lanes"] / N_LANES
    else:
        raise ValueError("SAE CSV needs either coverage or unique_valid_lanes.")

if "topk" not in sae.columns:
    sae["topk"] = np.nan

required_sae = [
    "capacity",
    "topk",
    "coverage",
    "lane_mass_ratio",
    "reconstruction_mse",
    "dead_features",
    "redundant_valid_features",
]
missing_sae = [c for c in required_sae if c not in sae.columns]
if missing_sae:
    raise ValueError(f"SAE CSV is missing required columns {missing_sae}. Columns found: {list(sae.columns)}")

sae["method"] = "SAE"

sae_keep = sae[
    [
        "method",
        "capacity",
        "topk",
        "coverage",
        "lane_mass_ratio",
        "reconstruction_mse",
        "dead_features",
        "redundant_valid_features",
    ]
].copy()

sae_keep.head()

In [ ]:
# Combine methods and compute structure-quality score

df = pd.concat([nmf_keep, sae_keep], ignore_index=True)

# Ensure numeric types.
for col in [
    "capacity",
    "topk",
    "coverage",
    "lane_mass_ratio",
    "reconstruction_mse",
    "dead_features",
    "redundant_valid_features",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Provisional structure-quality score:
# high quality requires coverage, lane alignment, low redundancy, and low dead features.
df["structure_quality"] = (
    df["coverage"].clip(lower=0, upper=1)
    * df["lane_mass_ratio"].clip(lower=0)
    * (1 / (1 + df["redundant_valid_features"].fillna(0)))
    * (1 / (1 + df["dead_features"].fillna(0)))
)

def classify(row):
    if row["coverage"] >= 0.99 and row["lane_mass_ratio"] >= 0.99 and row["dead_features"] == 0:
        return "recovered"
    if row["coverage"] < 0.75:
        return "fragmented"
    if row["dead_features"] > 0 or row["redundant_valid_features"] > 0:
        return "diluted"
    return "partial"

df["regime"] = df.apply(classify, axis=1)

df.to_csv("data/coverage_phase_diagram.csv", index=False)
print("Saved: data/coverage_phase_diagram.csv")

df.head()

In [ ]:
# Figure 1 — Coverage phase diagram

fig, ax = plt.subplots(figsize=(8, 5))

# NMF
nmf_plot = df[df["method"] == "NMF"].sort_values("capacity")
ax.plot(
    nmf_plot["capacity"],
    nmf_plot["coverage"],
    marker="o",
    label="NMF",
)

# SAE by top-k
sae_plot = df[df["method"] == "SAE"].copy()
for topk, group in sae_plot.groupby("topk"):
    group = group.sort_values("capacity")
    label = f"SAE top-k={int(topk)}" if pd.notna(topk) else "SAE"
    ax.plot(
        group["capacity"],
        group["coverage"],
        marker="o",
        label=label,
    )

ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.6)
ax.set_title("Residue-lane coverage by method and capacity")
ax.set_xlabel("Capacity (components / dictionary features)")
ax.set_ylabel("Coverage of 8 valid mod30 lanes")
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(True, alpha=0.25)

save_svg(fig, "coverage_phase_diagram")
plt.show()

In [ ]:
# Figure 2 — Alignment phase diagram

fig, ax = plt.subplots(figsize=(8, 5))

nmf_plot = df[df["method"] == "NMF"].sort_values("capacity")
ax.plot(
    nmf_plot["capacity"],
    nmf_plot["lane_mass_ratio"],
    marker="o",
    label="NMF",
)

for topk, group in sae_plot.groupby("topk"):
    group = group.sort_values("capacity")
    label = f"SAE top-k={int(topk)}" if pd.notna(topk) else "SAE"
    ax.plot(
        group["capacity"],
        group["lane_mass_ratio"],
        marker="o",
        label=label,
    )

ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.6)
ax.set_title("Lane-mass alignment by method and capacity")
ax.set_xlabel("Capacity (components / dictionary features)")
ax.set_ylabel("Mean mass on valid residue lanes")
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(True, alpha=0.25)

save_svg(fig, "alignment_phase_diagram")
plt.show()

In [ ]:
# Figure 3 — Cost vs structure quality

fig, ax = plt.subplots(figsize=(8, 5))

for method, group in df.groupby("method"):
    ax.scatter(
        group["reconstruction_mse"],
        group["structure_quality"],
        s=40 + 5 * group["capacity"],
        alpha=0.75,
        label=method,
    )

for _, row in df.iterrows():
    if row["method"] == "NMF" and row["capacity"] in [1, 8, 12]:
        ax.annotate(
            f'k={int(row["capacity"])}',
            (row["reconstruction_mse"], row["structure_quality"]),
            fontsize=8,
            xytext=(4, 4),
            textcoords="offset points",
        )

ax.set_title("Reconstruction cost vs structural quality")
ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Structure-quality score")
ax.legend()
ax.grid(True, alpha=0.25)

save_svg(fig, "cost_vs_structure_quality")
plt.show()

In [ ]:
# Figure 4 — Method regime map

fig, ax = plt.subplots(figsize=(8, 5))

regimes = sorted(df["regime"].unique())
markers = {
    "recovered": "o",
    "partial": "s",
    "fragmented": "^",
    "diluted": "x",
}

for regime in regimes:
    group = df[df["regime"] == regime]
    ax.scatter(
        group["capacity"],
        group["structure_quality"],
        marker=markers.get(regime, "o"),
        s=80,
        alpha=0.8,
        label=regime,
    )

for _, row in df.iterrows():
    label = row["method"]
    if row["method"] == "SAE" and pd.notna(row["topk"]):
        label = f'SAE k={int(row["topk"])}'
    ax.annotate(
        label,
        (row["capacity"], row["structure_quality"]),
        fontsize=7,
        xytext=(4, 3),
        textcoords="offset points",
        alpha=0.75,
    )

ax.set_title("Representation regimes across capacity")
ax.set_xlabel("Capacity (components / dictionary features)")
ax.set_ylabel("Structure-quality score")
ax.set_ylim(0, max(1.05, float(df["structure_quality"].max()) * 1.1))
ax.legend(title="Regime")
ax.grid(True, alpha=0.25)

save_svg(fig, "method_regime_map")
plt.show()

In [ ]:
# Method summary table

method_summary = (
    df.groupby("method")
    .agg(
        best_structure_quality=("structure_quality", "max"),
        best_coverage=("coverage", "max"),
        best_lane_mass_ratio=("lane_mass_ratio", "max"),
        min_reconstruction_mse=("reconstruction_mse", "min"),
        max_dead_features=("dead_features", "max"),
        max_redundant_valid_features=("redundant_valid_features", "max"),
    )
    .reset_index()
)

method_summary.to_csv("data/method_comparison_summary.csv", index=False)
print("Saved: data/method_comparison_summary.csv")

method_summary

## Paper claim

Across matched residue-manifold data, reconstruction error, capacity, and structural recovery separate into distinct regimes. Compact NMF recovery occupies a high-coverage/high-alignment region, while sparse autoencoder configurations can occupy lower-structure or diluted regimes even with increased capacity.

This prepares Notebook 06, where the provisional structure-quality score can be formalized as a CGCS-style metric.

In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "05_coverage_phase_diagram_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)